In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import math

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 1. 数据读取
print("正在读取数据...")
try:
    df = pd.read_csv('2020出租车12月7-13号数据.csv', encoding='utf-8', low_memory=False)
except:
    try:
        df = pd.read_csv('2020出租车12月7-13号数据.csv', encoding='gbk', low_memory=False)
    except:
        df = pd.read_csv('2020出租车12月7-13号数据.csv', encoding='latin1', low_memory=False)

print(f"数据形状: {df.shape}")

# 2. 列名映射
print("\n=== 应用列名映射 ===")
column_mapping = {
    'tm_on': '上车时间',
    'tm_off': '下车时间', 
    'ID': '订单ID',
    'CAR_ID': '车辆ID',
    'GET_OFF_TIME+0': '下车时间_备份',
    'GET_ON_TIME+0': '上车时间_备份',
    'MILEAGE': '行驶里程',
    'MONEY': '订单金额',
    'WAIT_TIME': '等待时长',
    'COMPANY_ID': '公司ID',
    'ON_LON': '上车经度',
    'ON_LAT': '上车纬度',
    'OFF_LON': '下车经度',
    'OFF_LAT': '下车纬度',
    'TRAVEL_TIME': '行驶时长',
    'tm_on_day': '天'
}

df.rename(columns=column_mapping, inplace=True)
print(f"重命名后的列名: {df.columns.tolist()}")

# 3. 数据处理
print("\n=== 数据处理 ===")

# 3.1 时间列处理
time_cols = [col for col in df.columns if '时间' in col]
for col in time_cols:
    try:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    except:
        pass

# 3.2 数值列处理
numeric_cols = ['行驶里程', '订单金额', '等待时长', '行驶时长', '天']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 4. 基础特征工程
print("\n=== 基础特征工程 ===")

# 4.1 时间特征
if '上车时间' in df.columns:
    df['上车小时'] = df['上车时间'].dt.hour
    df['上车星期'] = df['上车时间'].dt.dayofweek
    df['上车日期'] = df['上车时间'].dt.date
    
    # 时间段分类
    def get_time_period(hour):
        if 0 <= hour < 6:
            return '深夜'
        elif 6 <= hour < 9:
            return '早高峰'
        elif 9 <= hour < 12:
            return '上午'
        elif 12 <= hour < 14:
            return '午间'
        elif 14 <= hour < 17:
            return '下午'
        elif 17 <= hour < 19:
            return '晚高峰'
        else:
            return '夜间'
    
    df['时间段'] = df['上车小时'].apply(get_time_period)
    
    if '下车时间' in df.columns:
        df['行程时长_分钟'] = (df['下车时间'] - df['上车时间']).dt.total_seconds() / 60

# 4.2 经济特征
if '订单金额' in df.columns and '行驶里程' in df.columns:
    df['单价_元每公里'] = df['订单金额'] / df['行驶里程']
    df['单价_元每公里'] = df['单价_元每公里'].replace([np.inf, -np.inf], np.nan)

# 4.3 效率特征
if '行驶里程' in df.columns and '行程时长_分钟' in df.columns:
    df['平均速度_km_h'] = df['行驶里程'] / (df['行程时长_分钟'] / 60)
    df['平均速度_km_h'] = df['平均速度_km_h'].replace([np.inf, -np.inf], np.nan)

# 5. 生成图表（稳定版本，GitHub友好）
print("\n" + "="*60)
print("生成稳定版分析图表")
print("="*60)

# 5.1 热力图：小时-星期订单密度
print("\n1. 生成热力图：小时-星期订单密度")
if all(col in df.columns for col in ['上车小时', '上车星期']):
    heatmap_data = df.pivot_table(
        index='上车星期', 
        columns='上车小时', 
        values='订单ID',
        aggfunc='count',
        fill_value=0
    )
    
    plt.figure(figsize=(16, 8))
    sns.heatmap(heatmap_data, cmap='YlOrRd', annot=False, fmt='.0f', 
                cbar_kws={'label': '订单数量'})
    plt.title('小时-星期订单密度热力图', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('小时', fontsize=12)
    plt.ylabel('星期', fontsize=12)
    weekday_names = ['周一', '周二', '周三', '周四', '周五', '周六', '周日']
    plt.yticks(np.arange(7) + 0.5, weekday_names, rotation=0)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig('heatmap_hour_week.png', dpi=300, bbox_inches='tight')
    plt.show()

# 5.2 堆积面积图：各时段订单构成
print("\n2. 生成堆积面积图：各时段订单构成")
if '上车小时' in df.columns and '订单金额' in df.columns:
    bins = [0, 10, 20, 30, 50, 100, float('inf')]
    labels = ['0-10元', '10-20元', '20-30元', '30-50元', '50-100元', '100元以上']
    df['价格分段'] = pd.cut(df['订单金额'], bins=bins, labels=labels)
    
    time_price_data = df.pivot_table(
        index='上车小时',
        columns='价格分段',
        values='订单ID',
        aggfunc='count',
        fill_value=0
    )
    
    plt.figure(figsize=(14, 8))
    time_price_data.plot(kind='area', stacked=True, alpha=0.7)
    plt.title('各小时订单价格分段堆积图', fontsize=16, fontweight='bold')
    plt.xlabel('小时', fontsize=12)
    plt.ylabel('订单数量', fontsize=12)
    plt.legend(title='价格分段', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('area_price_hour.png', dpi=300, bbox_inches='tight')
    plt.show()

# 5.3 小提琴图：不同时间段的速度分布
print("\n3. 生成小提琴图：不同时间段的速度分布")
if all(col in df.columns for col in ['时间段', '平均速度_km_h']):
    speed_data = df[df['平均速度_km_h'].notna()].copy()
    q1 = speed_data['平均速度_km_h'].quantile(0.01)
    q3 = speed_data['平均速度_km_h'].quantile(0.99)
    speed_data = speed_data[(speed_data['平均速度_km_h'] >= q1) & 
                           (speed_data['平均速度_km_h'] <= q3)]
    
    plt.figure(figsize=(14, 8))
    sns.violinplot(x='时间段', y='平均速度_km_h', data=speed_data,
                  order=['深夜', '早高峰', '上午', '午间', '下午', '晚高峰', '夜间'],
                  palette='Set2', inner='quartile')
    plt.title('各时间段平均速度分布小提琴图', fontsize=16, fontweight='bold')
    plt.xlabel('时间段', fontsize=12)
    plt.ylabel('平均速度 (km/h)', fontsize=12)
    plt.axhline(y=30, color='red', linestyle='--', alpha=0.5, label='30km/h参考线')
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('violin_speed_period.png', dpi=300, bbox_inches='tight')
    plt.show()

# 5.4 日历热力图：每日订单趋势
print("\n4. 生成日历热力图：每日订单趋势")
if '上车日期' in df.columns:
    daily_orders = df.groupby('上车日期')['订单ID'].count().reset_index()
    daily_orders.columns = ['日期', '订单数']
    daily_orders['日期'] = pd.to_datetime(daily_orders['日期'])
    daily_orders['星期'] = daily_orders['日期'].dt.dayofweek
    daily_orders['周数'] = daily_orders['日期'].dt.isocalendar().week
    
    week_calendar = daily_orders.pivot_table(
        index='星期',
        columns='周数',
        values='订单数',
        aggfunc='sum',
        fill_value=0
    )
    
    plt.figure(figsize=(12, 6))
    sns.heatmap(week_calendar, cmap='RdYlGn', annot=True, fmt='.0f',
                cbar_kws={'label': '订单数量'}, linewidths=1, linecolor='white')
    plt.title('周视图日历热力图', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('周数', fontsize=12)
    plt.ylabel('星期', fontsize=12)
    weekday_names = ['周一', '周二', '周三', '周四', '周五', '周六', '周日']
    plt.yticks(np.arange(7) + 0.5, weekday_names, rotation=0)
    plt.tight_layout()
    plt.savefig('heatmap_calendar.png', dpi=300, bbox_inches='tight')
    plt.show()

# 5.5 时间序列分解图
print("\n5. 生成时间序列分解图")
if '上车时间' in df.columns:
    df['时间小时'] = df['上车时间'].dt.floor('H')
    hourly_series = df.groupby('时间小时')['订单ID'].count()
    hourly_series = hourly_series.asfreq('H', fill_value=0)
    window_size = 24
    trend = hourly_series.rolling(window=window_size, center=True).mean()
    daily_pattern = hourly_series.groupby(hourly_series.index.hour).mean()
    residual = hourly_series - trend

    fig, axes = plt.subplots(4, 1, figsize=(16, 12))
    axes[0].plot(hourly_series.index, hourly_series.values, color='blue', linewidth=1, alpha=0.7)
    axes[0].set_title('原始时间序列：每小时订单数', fontsize=14)
    axes[0].set_ylabel('订单数', fontsize=12)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(trend.index, trend.values, color='red', linewidth=2)
    axes[1].set_title('趋势成分（24小时移动平均）', fontsize=14)
    axes[1].set_ylabel('订单数', fontsize=12)
    axes[1].grid(True, alpha=0.3)

    axes[2].bar(range(24), daily_pattern.values, color='green', alpha=0.7)
    axes[2].set_title('季节性成分（每日模式）', fontsize=14)
    axes[2].set_xlabel('小时', fontsize=12)
    axes[2].set_ylabel('平均订单数', fontsize=12)
    axes[2].set_xticks(range(0, 24, 3))
    axes[2].grid(True, alpha=0.3)

    axes[3].plot(residual.index, residual.values, color='purple', linewidth=0.5, alpha=0.7)
    axes[3].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[3].set_title('残差成分', fontsize=14)
    axes[3].set_xlabel('时间', fontsize=12)
    axes[3].set_ylabel('订单数', fontsize=12)
    axes[3].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('ts_decompose.png', dpi=300, bbox_inches='tight')
    plt.show()

# 5.6 核密度估计图：联合分布
print("\n6. 生成核密度估计图：联合分布")
if all(col in df.columns for col in ['订单金额', '行驶里程']):
    sample_size = min(5000, len(df))
    kde_data = df[['订单金额', '行驶里程']].dropna().sample(n=sample_size, random_state=42)
    for col in kde_data.columns:
        q1 = kde_data[col].quantile(0.01)
        q3 = kde_data[col].quantile(0.99)
        kde_data = kde_data[(kde_data[col] >= q1) & (kde_data[col] <= q3)]

    plt.figure(figsize=(12, 10))
    g = sns.jointplot(x='行驶里程', y='订单金额', data=kde_data, 
                     kind='kde', fill=True, cmap='Blues', height=10,
                     marginal_kws=dict(fill=True, color='steelblue'))
    g.fig.suptitle('订单金额与行驶里程的联合核密度估计', fontsize=16, fontweight='bold', y=1.02)
    z = np.polyfit(kde_data['行驶里程'], kde_data['订单金额'], 1)
    p = np.poly1d(z)
    x_range = np.linspace(kde_data['行驶里程'].min(), kde_data['行驶里程'].max(), 100)
    g.ax_joint.plot(x_range, p(x_range), 'r--', linewidth=2, 
                   label=f'趋势线: y = {z[0]:.3f}x + {z[1]:.2f}')
    g.ax_joint.legend()
    g.ax_joint.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('kde_joint.png', dpi=300, bbox_inches='tight')
    plt.show()

# 5.7 箱线图组合：多维度对比
print("\n7. 生成箱线图组合：多维度对比")
if all(col in df.columns for col in ['上车星期', '订单金额', '行驶里程', '行程时长_分钟', '平均速度_km_h']):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    data_amount = df[['上车星期', '订单金额']].dropna()
    q1 = data_amount['订单金额'].quantile(0.01)
    q3 = data_amount['订单金额'].quantile(0.99)
    data_amount = data_amount[(data_amount['订单金额'] >= q1) & (data_amount['订单金额'] <= q3)]
    axes[0, 0].boxplot([data_amount[data_amount['上车星期'] == i]['订单金额'] for i in range(7)])
    axes[0, 0].set_title('各星期订单金额分布', fontsize=14)
    axes[0, 0].set_xlabel('星期', fontsize=12)
    axes[0, 0].set_ylabel('订单金额(元)', fontsize=12)
    axes[0, 0].set_xticklabels(['周一', '周二', '周三', '周四', '周五', '周六', '周日'])
    axes[0, 0].grid(True, alpha=0.3)

    data_mileage = df[['上车星期', '行驶里程']].dropna()
    q1 = data_mileage['行驶里程'].quantile(0.01)
    q3 = data_mileage['行驶里程'].quantile(0.99)
    data_mileage = data_mileage[(data_mileage['行驶里程'] >= q1) & (data_mileage['行驶里程'] <= q3)]
    axes[0, 1].boxplot([data_mileage[data_mileage['上车星期'] == i]['行驶里程'] for i in range(7)])
    axes[0, 1].set_title('各星期行驶里程分布', fontsize=14)
    axes[0, 1].set_xlabel('星期', fontsize=12)
    axes[0, 1].set_ylabel('行驶里程(km)', fontsize=12)
    axes[0, 1].set_xticklabels(['周一', '周二', '周三', '周四', '周五', '周六', '周日'])
    axes[0, 1].grid(True, alpha=0.3)

    data_duration = df[['上车星期', '行程时长_分钟']].dropna()
    q1 = data_duration['行程时长_分钟'].quantile(0.01)
    q3 = data_duration['行程时长_分钟'].quantile(0.99)
    data_duration = data_duration[(data_duration['行程时长_分钟'] >= q1) & (data_duration['行程时长_分钟'] <= q3)]
    axes[1, 0].boxplot([data_duration[data_duration['上车星期'] == i]['行程时长_分钟'] for i in range(7)])
    axes[1, 0].set_title('各星期行程时长分布', fontsize=14)
    axes[1, 0].set_xlabel('星期', fontsize=12)
    axes[1, 0].set_ylabel('行程时长(分钟)', fontsize=12)
    axes[1, 0].set_xticklabels(['周一', '周二', '周三', '周四', '周五', '周六', '周日'])
    axes[1, 0].grid(True, alpha=0.3)

    data_speed = df[['上车星期', '平均速度_km_h']].dropna()
    q1 = data_speed['平均速度_km_h'].quantile(0.01)
    q3 = data_speed['平均速度_km_h'].quantile(0.99)
    data_speed = data_speed[(data_speed['平均速度_km_h'] >= q1) & (data_speed['平均速度_km_h'] <= q3)]
    axes[1, 1].boxplot([data_speed[data_speed['上车星期'] == i]['平均速度_km_h'] for i in range(7)])
    axes[1, 1].set_title('各星期平均速度分布', fontsize=14)
    axes[1, 1].set_xlabel('星期', fontsize=12)
    axes[1, 1].set_ylabel('平均速度(km/h)', fontsize=12)
    axes[1, 1].set_xticklabels(['周一', '周二', '周三', '周四', '周五', '周六', '周日'])
    axes[1, 1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('boxplot_week.png', dpi=300, bbox_inches='tight')
    plt.show()

print("\n✅ 稳定版图表生成完毕！")